# Embabel Examples Replicated with LangGoap

This notebook replicates key patterns from [Embabel](https://github.com/embabel/embabel-agent)
using LangGoap. Embabel defines agents using GOAP planning with type-safe
preconditions and effects. LangGoap brings the same patterns to the LangChain ecosystem.

## Examples

1. **Star News Finder** — Multi-step LLM pipeline with web search
2. **Meal Preparation** — Parallel preconditions merging at a goal action
3. **Write and Review** — Iterative refinement via replanning
4. **Fact Checker** — Multi-step verification pipeline
5. **Cost-Based Planning** — A\* prefers cheap actions over expensive ones

In [1]:
from typing import Any

from langgoap import ActionSpec, GoalSpec, GoapGraph, ReplanStrategy

---

## 1. Star News Finder

The original Embabel pattern chains:

```
extractPerson(UserInput) → Person
extractStarPerson(Person) → StarPerson
retrieveHoroscope(StarPerson) → Horoscope
findNewsStories(StarPerson, Horoscope) → RelevantNewsStories
starNewsWriteup(StarPerson, Horoscope, News) → Writeup  @AchievesGoal
```

In LangGoap, the A\* planner discovers this sequence from preconditions/effects.

In [2]:
def extract_person(ws: dict[str, Any]) -> dict[str, Any]:
    user_input = ws.get("user_input", "")
    return {"has_person": True, "person": {"name": "Alice", "extracted_from": user_input}}


def extract_star_sign(ws: dict[str, Any]) -> dict[str, Any]:
    person = ws.get("person", {})
    return {"has_star_person": True, "star_person": {**person, "sign": "Aries"}}


def retrieve_horoscope(ws: dict[str, Any]) -> dict[str, Any]:
    star_person = ws.get("star_person", {})
    sign = star_person.get("sign", "Unknown")
    return {"has_horoscope": True, "horoscope": f"Today {sign} will experience great fortune in technology."}


def find_news_stories(ws: dict[str, Any]) -> dict[str, Any]:
    person = ws.get("star_person", {})
    name = person.get("name", "")
    return {
        "has_news": True,
        "news_stories": [
            f"{name} featured in AI conference keynote",
            f"New developments in {person.get('sign', '')} season",
        ],
    }


def star_news_writeup(ws: dict[str, Any]) -> dict[str, Any]:
    person = ws.get("star_person", {})
    horoscope = ws.get("horoscope", "")
    news = ws.get("news_stories", [])
    return {
        "writeup_complete": True,
        "writeup": (
            f"Star News for {person.get('name', '')}: "
            f"{horoscope} "
            f"In the news: {'; '.join(news)}"
        ),
    }


star_news_actions = [
    ActionSpec(name="extract_person", preconditions={"has_user_input": True}, effects={"has_person": True}, execute=extract_person),
    ActionSpec(name="extract_star_sign", preconditions={"has_person": True}, effects={"has_star_person": True}, execute=extract_star_sign),
    ActionSpec(name="retrieve_horoscope", preconditions={"has_star_person": True}, effects={"has_horoscope": True}, execute=retrieve_horoscope),
    ActionSpec(name="find_news_stories", preconditions={"has_star_person": True}, effects={"has_news": True}, cost=2.0, execute=find_news_stories),
    ActionSpec(name="star_news_writeup", preconditions={"has_horoscope": True, "has_news": True}, effects={"writeup_complete": True}, execute=star_news_writeup),
]

In [3]:
result = GoapGraph(actions=star_news_actions).invoke(
    goal=GoalSpec(conditions={"writeup_complete": True}),
    world_state={"has_user_input": True, "user_input": "Tell me about Alice who is an Aries"},
)

print(f"Status: {result['status']}")
print(f"Writeup: {result['world_state']['writeup']}")
print()
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Pipeline: {' → '.join(successful)}")

Status: goal_achieved
Writeup: Star News for Alice: Today Aries will experience great fortune in technology. In the news: Alice featured in AI conference keynote; New developments in Aries season

Pipeline: extract_person → extract_star_sign → retrieve_horoscope → find_news_stories → star_news_writeup


### Partial Goal: Horoscope Only

The planner stops at `retrieve_horoscope` — no news search or writeup needed.

In [4]:
result = GoapGraph(actions=star_news_actions).invoke(
    goal=GoalSpec(conditions={"has_horoscope": True}),
    world_state={"has_user_input": True},
)

successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Actions: {successful}")
print(f"Writeup skipped: {'star_news_writeup' not in successful}")

Actions: ['extract_person', 'extract_star_sign', 'retrieve_horoscope']
Writeup skipped: True


---

## 2. Meal Preparation

Classic GOAP pattern: two independent precondition paths merge at the final action.

```
chooseCook(UserInput)  → Cook
takeOrder(UserInput)   → Order
prepareMeal(Cook, Order) → Meal  @AchievesGoal
```

Both `choose_cook` and `take_order` share the same precondition but produce
different effects. `prepare_meal` requires both.

In [5]:
def choose_cook(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_cook": True, "cook": {"name": "Chef Marie", "specialty": "French cuisine"}}


def take_order(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_order": True, "order": {"dish": "Coq au Vin", "special_requests": "no mushrooms"}}


def prepare_meal(ws: dict[str, Any]) -> dict[str, Any]:
    cook = ws.get("cook", {})
    order = ws.get("order", {})
    return {
        "meal_ready": True,
        "meal": {
            "dish": order.get("dish", ""),
            "prepared_by": cook.get("name", ""),
            "quality": "excellent",
        },
    }


meal_actions = [
    ActionSpec(name="choose_cook", preconditions={"has_user_input": True}, effects={"has_cook": True}, execute=choose_cook),
    ActionSpec(name="take_order", preconditions={"has_user_input": True}, effects={"has_order": True}, execute=take_order),
    ActionSpec(name="prepare_meal", preconditions={"has_cook": True, "has_order": True}, effects={"meal_ready": True}, execute=prepare_meal),
]

In [6]:
result = GoapGraph(actions=meal_actions).invoke(
    goal=GoalSpec(conditions={"meal_ready": True}),
    world_state={"has_user_input": True},
)

print(f"Status: {result['status']}")
print(f"Meal: {result['world_state']['meal']}")

successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Actions: {successful}")
# prepare_meal must come after both prerequisites
print(f"prepare_meal after choose_cook: {successful.index('prepare_meal') > successful.index('choose_cook')}")
print(f"prepare_meal after take_order: {successful.index('prepare_meal') > successful.index('take_order')}")

Status: goal_achieved
Meal: {'dish': 'Coq au Vin', 'prepared_by': 'Chef Marie', 'quality': 'excellent'}
Actions: ['choose_cook', 'take_order', 'prepare_meal']
prepare_meal after choose_cook: True
prepare_meal after take_order: True


### Partial Prerequisites

When the cook is already chosen, only `take_order → prepare_meal` are needed.

In [7]:
result = GoapGraph(actions=meal_actions).invoke(
    goal=GoalSpec(conditions={"meal_ready": True}),
    world_state={
        "has_user_input": True,
        "has_cook": True,
        "cook": {"name": "Chef Pierre", "specialty": "Italian"},
    },
)

successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Actions: {successful}")
print(f"choose_cook skipped: {'choose_cook' not in successful}")

Actions: ['take_order', 'prepare_meal']
choose_cook skipped: True


### Goal Already Satisfied

When the meal is already ready, no actions are needed.

In [8]:
result = GoapGraph(actions=meal_actions).invoke(
    goal=GoalSpec(conditions={"meal_ready": True}),
    world_state={"meal_ready": True},
)

successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Status: {result['status']}")
print(f"Actions executed: {len(successful)}")

Status: goal_achieved
Actions executed: 0


---

## 3. Write and Review

The Embabel pattern uses assess/revise loops for iterative refinement:

```
craftStory(UserInput) → Story
assess(Story) → Assessment (accept/reject)
If rejected: revise(Story, Feedback) → Story (loop)
If accepted: review(Story) → ReviewedStory  @AchievesGoal
```

In LangGoap, iterative refinement happens via **replanning**. When the reviewer
sets `review_complete=False` (deviating from the plan's expected `True`),
the observer triggers replanning from the current state.

In [9]:
def craft_story(ws: dict[str, Any]) -> dict[str, Any]:
    user_input = ws.get("user_input", "a space adventure")
    return {"has_story": True, "story": f"Once upon a time, in a galaxy far away, {user_input}..."}


def review_story(ws: dict[str, Any]) -> dict[str, Any]:
    story = ws.get("story", "")
    return {
        "review_complete": True,
        "reviewed_story": {
            "story": story,
            "review": "A captivating narrative with excellent pacing.",
            "reviewer": "NYT Book Review",
        },
    }


write_review_actions = [
    ActionSpec(name="craft_story", preconditions={"has_user_input": True}, effects={"has_story": True}, execute=craft_story),
    ActionSpec(name="review_story", preconditions={"has_story": True}, effects={"review_complete": True}, execute=review_story),
]

result = GoapGraph(actions=write_review_actions).invoke(
    goal=GoalSpec(conditions={"review_complete": True}),
    world_state={"has_user_input": True, "user_input": "a brave astronaut explored Mars"},
)

print(f"Status: {result['status']}")
ws = result["world_state"]
print(f"Story: {ws['story']}")
print(f"Review: {ws['reviewed_story']['review']}")
print(f"Reviewer: {ws['reviewed_story']['reviewer']}")

Status: goal_achieved
Story: Once upon a time, in a galaxy far away, a brave astronaut explored Mars...
Review: A captivating narrative with excellent pacing.
Reviewer: NYT Book Review


### Revision via Replanning

When the reviewer rejects the story, it sets `review_complete=False` and
`has_story=False`, deviating from expectations. The observer detects this
and triggers replanning, which re-includes `craft_story` in the new plan.

In [10]:
craft_count = {"count": 0}


def craft_with_revision(ws: dict[str, Any]) -> dict[str, Any]:
    craft_count["count"] += 1
    user_input = ws.get("user_input", "")
    if craft_count["count"] == 1:
        return {"has_story": True, "story": "A very short and unimaginative tale."}
    return {"has_story": True, "story": f"[REVISED] An epic saga: {user_input}"}


def review_with_standards(ws: dict[str, Any]) -> dict[str, Any]:
    story = ws.get("story", "")
    if "REVISED" not in story:
        # Reject: clear has_story so replanner re-includes craft_story
        return {"review_complete": False, "has_story": False}
    return {
        "review_complete": True,
        "reviewed_story": {
            "story": story,
            "review": "Much improved! Compelling narrative.",
            "reviewer": "NYT Book Review",
        },
    }


revision_actions = [
    ActionSpec(name="craft_story", preconditions={"has_user_input": True}, effects={"has_story": True}, execute=craft_with_revision),
    ActionSpec(name="review_story", preconditions={"has_story": True}, effects={"review_complete": True}, execute=review_with_standards),
]

result = GoapGraph(actions=revision_actions).invoke(
    goal=GoalSpec(
        conditions={"review_complete": True},
        replan_strategy=ReplanStrategy.ON_DEVIATION,
    ),
    world_state={"has_user_input": True, "user_input": "dragons and knights"},
)

print(f"Status: {result['status']}")
print(f"Replans: {result['replan_count']}")
print(f"Craft attempts: {craft_count['count']}")
print(f"Final story: {result['world_state']['story']}")

Status: goal_achieved
Replans: 1
Craft attempts: 2
Final story: [REVISED] An epic saga: dragons and knights


---

## 4. Fact Checker

The Embabel pattern chains extraction → rationalization → verification:

```
extractAssertions(Content) → FactualAssertions
rationalizeAssertions(FactualAssertions) → RationalizedAssertions
checkFacts(RationalizedAssertions) → FactCheck  @AchievesGoal
```

In [11]:
def extract_assertions(ws: dict[str, Any]) -> dict[str, Any]:
    content = ws.get("content", "")
    return {
        "has_assertions": True,
        "assertions": [
            {"claim": "Python was created in 1991", "source": content[:50]},
            {"claim": "LangChain was released in 2022", "source": content[:50]},
        ],
    }


def rationalize_assertions(ws: dict[str, Any]) -> dict[str, Any]:
    assertions = ws.get("assertions", [])
    return {
        "has_rationalized": True,
        "rationalized_assertions": [{**a, "importance": "high"} for a in assertions],
    }


def check_facts(ws: dict[str, Any]) -> dict[str, Any]:
    assertions = ws.get("rationalized_assertions", [])
    checks = [{"claim": a["claim"], "verified": True, "confidence": 0.95} for a in assertions]
    return {
        "fact_check_complete": True,
        "fact_check": {
            "total_claims": len(checks),
            "verified": sum(1 for c in checks if c["verified"]),
            "checks": checks,
        },
    }


fact_check_actions = [
    ActionSpec(name="extract_assertions", preconditions={"has_content": True}, effects={"has_assertions": True}, execute=extract_assertions),
    ActionSpec(name="rationalize_assertions", preconditions={"has_assertions": True}, effects={"has_rationalized": True}, execute=rationalize_assertions),
    ActionSpec(name="check_facts", preconditions={"has_rationalized": True}, effects={"fact_check_complete": True}, execute=check_facts),
]

In [12]:
result = GoapGraph(actions=fact_check_actions).invoke(
    goal=GoalSpec(conditions={"fact_check_complete": True}),
    world_state={
        "has_content": True,
        "content": "Python was created by Guido van Rossum in 1991. "
        "LangChain was released in 2022 for LLM app development.",
    },
)

fc = result["world_state"]["fact_check"]
print(f"Status: {result['status']}")
print(f"Claims checked: {fc['total_claims']}")
print(f"Verified: {fc['verified']}")
for check in fc["checks"]:
    print(f"  {check['claim']}: verified={check['verified']} (confidence={check['confidence']})")

successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"\nPipeline: {' → '.join(successful)}")

Status: goal_achieved
Claims checked: 2
Verified: 2
  Python was created in 1991: verified=True (confidence=0.95)
  LangChain was released in 2022: verified=True (confidence=0.95)

Pipeline: extract_assertions → rationalize_assertions → check_facts


---

## 5. Cost-Based Planning

Embabel uses `@Action(cost=100)` to mark expensive actions as last resort.
LangGoap's A\* planner minimizes total cost, so cheap actions are always preferred.

In [13]:
def cache_lookup(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_answer": True, "answer": "cached result", "source": "cache"}


def api_call(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_answer": True, "answer": "api result", "source": "api"}


cost_actions = [
    ActionSpec(
        name="cache_lookup",
        preconditions={"has_query": True},
        effects={"has_answer": True},
        cost=1.0,  # Cheap
        execute=cache_lookup,
    ),
    ActionSpec(
        name="api_call",
        preconditions={"has_query": True},
        effects={"has_answer": True},
        cost=100.0,  # Expensive: @Action(cost=100) equivalent
        execute=api_call,
    ),
]

result = GoapGraph(actions=cost_actions).invoke(
    goal=GoalSpec(conditions={"has_answer": True}),
    world_state={"has_query": True, "query": "What is GOAP?"},
)

ws = result["world_state"]
print(f"Source: {ws['source']} (cache preferred over API)")
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Action used: {successful[0]}")

Source: cache (cache preferred over API)
Action used: cache_lookup


### Multi-Step Cost Optimization

A\* optimizes total path cost across multiple steps.

In [14]:
def fast_extract(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_data": True, "data": "fast"}


def thorough_extract(ws: dict[str, Any]) -> dict[str, Any]:
    return {"has_data": True, "data": "thorough"}


def process(ws: dict[str, Any]) -> dict[str, Any]:
    return {"processed": True, "result": f"processed {ws.get('data', '')}"}


multi_cost_actions = [
    ActionSpec(name="fast_extract", preconditions={"has_input": True}, effects={"has_data": True}, cost=1.0, execute=fast_extract),
    ActionSpec(name="thorough_extract", preconditions={"has_input": True}, effects={"has_data": True}, cost=10.0, execute=thorough_extract),
    ActionSpec(name="process", preconditions={"has_data": True}, effects={"processed": True}, cost=1.0, execute=process),
]

result = GoapGraph(actions=multi_cost_actions).invoke(
    goal=GoalSpec(conditions={"processed": True}),
    world_state={"has_input": True},
)

print(f"Result: {result['world_state']['result']}")
print(f"Path: fast_extract(1) + process(1) = 2  vs  thorough_extract(10) + process(1) = 11")
successful = [h.action_name for h in result["execution_history"] if h.success]
print(f"Chosen path: {' → '.join(successful)}")

Result: processed fast
Path: fast_extract(1) + process(1) = 2  vs  thorough_extract(10) + process(1) = 11
Chosen path: fast_extract → process
